# FlexRank Checkpoint Save, Load, and Prune

A compact, model-configurable workflow for causal language models: load a pretrained model, optionally create a `FlexRankModel` with DataSVD, save and reload it through `from_pretrained`, prune it to requested profile sizes, deploy the pruned GAR form, and verify the reloaded checkpoint preserves loss and storage metadata.

Set `FLEXRANK_CHECKPOINT` in the parameter cell to skip local DataSVD/profile creation and start from an existing Hub or local FlexRank checkpoint.

## Imports and setup

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import os
import shutil
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "flextrain").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

from datasets import Dataset, IterableDataset
from transformers import TrainingArguments

from flexrank.layers import ODImpl, ODLinear
from flexrank.profiles import DPSearchAlgo
from flexrank.samplers.base_sampler import DeployMode
from flexrank.trainers import SVDType
from flextrain.dataset import load_dataset_from_hf
from flextrain.model import FlexRankModel, ModelWithProcessorAndMetric, load_model_from_hf, replace_conv1d_with_linear
from flextrain.utils.args import NLPDataArguments, NLPModelArguments, TaskName
from flextrain.utils.flexrank_utils import init_eval_trainer, init_flexrank_model
from flextrain.utils.utils import suppress_stdout
from safetensors.torch import save_file
import tempfile


class ExportNamespace(SimpleNamespace):
    def to_export_dict(self):
        return dict(vars(self))


def materialize_dataset(dataset):
    if isinstance(dataset, IterableDataset):
        return Dataset.from_list(list(dataset))
    return dataset

@suppress_stdout
def evaluate_model(model, eval_dataset, collator, processor):
    eval_trainer = init_eval_trainer(
        ModelWithProcessorAndMetric(model, processor, None),
        collator,
        eval_dataset,
        eval_args,
        deepcopy_model=False,
    )
    return eval_trainer.evaluate()


def extract_eval_loss(metrics):
    if "eval_loss" not in metrics:
        raise RuntimeError(f"Evaluation did not return eval_loss. Available metrics: {sorted(metrics)}")
    return float(metrics["eval_loss"])


def decomposed_safetensors_size(model, layer_names, prefix=""):
    tensors = {
        key: tensor.detach().cpu().contiguous()
        for key, tensor in model.state_dict().items()
        if any(key.startswith(f"{prefix}{name}.") for name in layer_names)
    }
    with tempfile.NamedTemporaryFile(suffix=".safetensors", delete=True) as tmp_file:
        save_file(tensors, tmp_file.name)
        return Path(tmp_file.name).stat().st_size

def odlinear_impl_counts(model):
    counts = {impl.value: 0 for impl in ODImpl}
    for module in model.modules():
        if isinstance(module, ODLinear):
            counts[module.impl.value] += 1
    return {key: value for key, value in counts.items() if value}



In [2]:
# Model and dataset
MODEL_NAME_OR_PATH = "openai-community/gpt2"  #@param {type:"string"}
EXCLUDE_LAYERS_NAMES = "lm_head"  #@param {type:"string"}

DATASET_PATH = "HuggingFaceFW/fineweb-edu"  #@param {type:"string"}
DATASET_NAME = "sample-10BT"  #@param {type:"string"}
SEQUENCE_LENGTH = 256  #@param {type:"integer"}
EVAL_DS_SIZE = 16  #@param {type:"integer"}
CALIB_DS_SIZE = 2  #@param {type:"integer"}

# Pruning and profiles
TEST_SIZE_RATIOS = "0.45, 0.65, 0.95"  #@param {type:"string"}
PROFILE_N_MODELS = 10  #@param {type:"integer"}
PROFILE_MIN_P = 0.1  #@param {type:"number"}
DEPLOY_MODE = "GAR"  #@param ["NO", "SVD", "GAR"]

# Hub/local checkpoint controls
# Set FLEXRANK_CHECKPOINT to "namespace/repo_name" to skip local DataSVD/profile creation.
FLEXRANK_CHECKPOINT = "riccardozaccone96/gpt2_datasvd_base"  #@param {type:"string"}

# Normalize form-like strings into the values used by later cells.
MODEL_TAG = MODEL_NAME_OR_PATH.split("/")[-1].replace("-", "_")
EXCLUDE_LAYERS_NAMES = [item.strip() for item in EXCLUDE_LAYERS_NAMES.split(",") if item.strip()]
DATASET_NAME = DATASET_NAME or None
TEST_SIZE_RATIOS = tuple(float(item.strip()) for item in TEST_SIZE_RATIOS.split(",") if item.strip())
DEPLOY_SIZE_RATIO = TEST_SIZE_RATIOS[-1]
DEPLOY_MODE = {mode.name: mode for mode in DeployMode}[DEPLOY_MODE.upper()]
FLEXRANK_CHECKPOINT = FLEXRANK_CHECKPOINT or None

# Saving directories
OUTPUT_ROOT = Path("outputs/notebooks")
BASE_SAVE_DIR = OUTPUT_ROOT / f"{MODEL_TAG}_base_demo"
SAVE_DIR = OUTPUT_ROOT / f"{MODEL_TAG}_flexrank_demo"
EVAL_DIR = OUTPUT_ROOT / f"{MODEL_TAG}_flexrank_eval"
PRUNED_SAVE_DIR = SAVE_DIR.parent / f"{SAVE_DIR.name}_{DEPLOY_MODE.value}_{DEPLOY_SIZE_RATIO:g}"

{
    "model": MODEL_NAME_OR_PATH,
    "flexrank_checkpoint": FLEXRANK_CHECKPOINT,
    "test_size_ratios": TEST_SIZE_RATIOS,
    "deployment_size_ratio": DEPLOY_SIZE_RATIO,
    "deploy_mode": DEPLOY_MODE.name,
    "save_dir": str(SAVE_DIR),
}


{'model': 'openai-community/gpt2',
 'flexrank_checkpoint': 'riccardozaccone96/gpt2_datasvd_base',
 'test_size_ratios': (0.45, 0.65, 0.95),
 'deployment_size_ratio': 0.95,
 'deploy_mode': 'GAR',
 'save_dir': 'outputs/notebooks/gpt2_flexrank_demo'}

## Obtain a FlexRankModel

### Simulate a pretrained model
Use this path if you don't have a local flexrank checkpoint and cannot access the HF hub.

Load the pretrained base model and a small tokenized FineWeb-Edu split through the same Hugging Face helpers used by `train.py`.


In [3]:
model_args = NLPModelArguments(
    model_name_or_path=MODEL_NAME_OR_PATH,
    trust_remote_code=True,
    use_auth_token=False,
    load_pretrained_model=True,
    _torch_dtype="torch.float32",
    cache_dir=None,
)
model_data = load_model_from_hf(TaskName.NLP, model_args)
replace_conv1d_with_linear(model_data.model)

data_args = NLPDataArguments(
    path=DATASET_PATH,
    name=DATASET_NAME,
    streaming=True,
    eval_ds_size=EVAL_DS_SIZE,
    calib_ds_size=CALIB_DS_SIZE,
    sequence_length=SEQUENCE_LENGTH,
    cache_file_names=None,
)
splits = load_dataset_from_hf(TaskName.NLP, data_args, model_data.proc, data_seed=0)
splits = SimpleNamespace(
    train=splits.train,
    val=materialize_dataset(splits.val),
    calib=materialize_dataset(splits.calib),
    collator=splits.collator,
)

eval_args = TrainingArguments(
    output_dir=EVAL_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    max_steps=1,
    learning_rate=0.0,
    report_to="none",
    disable_tqdm=True,
    remove_unused_columns=False,
    label_names=["labels"],
)

base_metrics = evaluate_model(model_data.model, splits.val, splits.collator, model_data.proc)

shutil.rmtree(BASE_SAVE_DIR, ignore_errors=True)
model_data.model.save_pretrained(BASE_SAVE_DIR)

base_metrics

2026-05-22 00:32:01 - INFO (flextrain.model.hf_nlp_model:38): loading base model openai-community/gpt2...
2026-05-22 00:32:01 - INFO (flextrain.model.hf_nlp_model:41): loading pretrained weights
2026-05-22 00:32:02 - INFO (flextrain.model.hf_nlp_model:75): Setting pad token to [EOS] token
/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized

{'eval_loss': 3.2873520851135254,
 'eval_runtime': 1.8117,
 'eval_samples_per_second': 8.832,
 'eval_steps_per_second': 4.416,
 'epoch': 0.125}

Initialize the FlexRank model by decomposing the source model with DataSVD on the calibration tokens, then wrap the decomposed model as `FlexRankModel`.


In [4]:
if FLEXRANK_CHECKPOINT is None:
    calib_trainer = init_eval_trainer(
        model_data,
        splits.collator,
        splits.calib,
        eval_args,
        deepcopy_model=False,
    )
    demo_args = SimpleNamespace(
        dataset=ExportNamespace(**data_args.to_export_dict()),
        decomposition=ExportNamespace(
            svd_type=SVDType.DataSVD,
            exclude_layers_names=EXCLUDE_LAYERS_NAMES,
            freeze_non_decomposed=False,
            max_svd_data_count=CALIB_DS_SIZE,
            gram_cache_device="cpu",
        ),
        profile_algo=ExportNamespace(classname="DPSearchAlgo"),
        sampler=ExportNamespace(classname="BaseSampler"),
        train=ExportNamespace(freeze_non_decomposed=False),
        model=ExportNamespace(**model_args.to_export_dict()),
    )

    flex_model_data, decomposed_params = init_flexrank_model(
        model_data,
        calib_trainer,
        demo_args,
    )
    flex_model = flex_model_data.model
else:
    decomposed_params = None
    flex_model = FlexRankModel.from_pretrained(FLEXRANK_CHECKPOINT)

flex_metrics = evaluate_model(flex_model, splits.val, splits.collator, model_data.proc)

{
    "source": "hub" if FLEXRANK_CHECKPOINT else "local_datasvd",
    "decomposed_layers": len(flex_model.decomposed_layer_names),
    "decomposed_params": decomposed_params,
    "sample_layers": sorted(flex_model.decomposed_layer_names)[:5],
    **flex_metrics,
}


/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'source': 'hub',
 'decomposed_layers': 48,
 'decomposed_params': None,
 'sample_layers': ['transformer.h.0.attn.c_attn',
  'transformer.h.0.attn.c_proj',
  'transformer.h.0.mlp.c_fc',
  'transformer.h.0.mlp.c_proj',
  'transformer.h.1.attn.c_attn'],
 'eval_loss': 3.287339687347412,
 'eval_runtime': 2.0862,
 'eval_samples_per_second': 7.669,
 'eval_steps_per_second': 3.835,
 'epoch': 0.125}

Model pruning (via `flex_model.reduce_size(...)`) requires profile metadata to select the proper submodel. For this compact demo, use `DPSearchAlgo` to create budget-aware profiles. In full experiments, run the Knowledge Consolidation phase to obtain a performant elastic model.

In [5]:
if FLEXRANK_CHECKPOINT is None:
    profile_trainer = init_eval_trainer(
        ModelWithProcessorAndMetric(flex_model.base_model, model_data.proc, None),
        splits.collator,
        splits.calib,
        eval_args,
        deepcopy_model=False,
    )

    dp_solution = DPSearchAlgo(
        evaluator=profile_trainer,
        n_models=PROFILE_N_MODELS,
        min_p=PROFILE_MIN_P
    ).solve()

    flex_model.profiles_data = dp_solution.to_profiles_data()


In [6]:
if FLEXRANK_CHECKPOINT is None:
    shutil.rmtree(SAVE_DIR, ignore_errors=True)
    flex_model.save_pretrained(SAVE_DIR)


### Load from Hub/local path

In [7]:
CHECKPOINT_PATH = FLEXRANK_CHECKPOINT or SAVE_DIR

loaded_model = FlexRankModel.from_pretrained(CHECKPOINT_PATH)
loaded_metrics = evaluate_model(loaded_model, splits.val, splits.collator, model_data.proc)

{
    "source": "hub" if FLEXRANK_CHECKPOINT else "local_datasvd",
    "base_model_type": loaded_model.config.base_model_type,
    "base_auto_class": loaded_model.config.base_auto_class,
    "decomposed_layers": len(loaded_model.decomposed_layer_names),
    **loaded_metrics,
}

/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'source': 'hub',
 'base_model_type': 'gpt2',
 'base_auto_class': 'AutoModelForCausalLM',
 'decomposed_layers': 48,
 'eval_loss': 3.287339687347412,
 'eval_runtime': 2.1072,
 'eval_samples_per_second': 7.593,
 'eval_steps_per_second': 3.796,
 'epoch': 0.125}

## Pruning the model

### Evaluating different submodels without physical pruning

In [8]:
pruned_eval_rows = []
for size_ratio in TEST_SIZE_RATIOS:
    loaded_model.reduce_size(size_ratio=size_ratio)
    metrics = evaluate_model(loaded_model, splits.val, splits.collator, model_data.proc)
    pruned_eval_rows.append(
        {
            "target_size_ratio": size_ratio,
            "virtual_size_ratio": loaded_model.virtual_size_ratio,
            "physical_size_ratio": loaded_model.physical_size_ratio,
            "eval_loss": extract_eval_loss(metrics),
        }
    )

pruned_eval_rows

/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/rzaccone/envs/flexrank/lib/p

[{'target_size_ratio': 0.45,
  'virtual_size_ratio': 0.4166373080397471,
  'physical_size_ratio': 1.0,
  'eval_loss': 7.1269659996032715},
 {'target_size_ratio': 0.65,
  'virtual_size_ratio': 0.6044083107497742,
  'physical_size_ratio': 1.0,
  'eval_loss': 6.108086585998535},
 {'target_size_ratio': 0.95,
  'virtual_size_ratio': 0.9007446928635953,
  'physical_size_ratio': 1.0,
  'eval_loss': 3.453774929046631}]

### Deploying a model
Prune to a requested size and physically deploy the decomposed weights. `DeployMode.NO` ("NO") keeps the profile virtual, `DeployMode.SVD` ("SVD") stores sliced SVD factors, and `DeployMode.GAR` ("GAR") stores GAR factors where supported; unsupported layers fall back to SVD with a warning.

In [9]:
loaded_model.reduce_size(size_ratio=DEPLOY_SIZE_RATIO, deploy_mode=DEPLOY_MODE)
deploy_metrics = evaluate_model(loaded_model, splits.val, splits.collator, model_data.proc)

{
    "target_size_ratio": DEPLOY_SIZE_RATIO,
    "virtual_size_ratio": loaded_model.virtual_size_ratio,
    "physical_size_ratio": loaded_model.physical_size_ratio,
    "eval_loss": extract_eval_loss(deploy_metrics),
    "odlinear_impl_counts": odlinear_impl_counts(loaded_model),
}

/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'target_size_ratio': 0.95,
 'virtual_size_ratio': 0.9007446928635953,
 'physical_size_ratio': 0.9007446928635953,
 'eval_loss': 3.45381498336792,
 'odlinear_impl_counts': {'gar': 48}}

### Saving the pruned model
Save the physically pruned model, compare its checkpoint weight size with the original save, then load it again to check that it works and has the same loss.

In [10]:
shutil.rmtree(PRUNED_SAVE_DIR, ignore_errors=True)
loaded_model.save_pretrained(PRUNED_SAVE_DIR)

reloaded_pruned_model = FlexRankModel.from_pretrained(PRUNED_SAVE_DIR)
reloaded_pruned_metrics = evaluate_model(reloaded_pruned_model, splits.val, splits.collator, model_data.proc)

layer_names = reloaded_pruned_model.decomposed_layer_names
base_disk_bytes = decomposed_safetensors_size(model_data.model, layer_names)
pruned_disk_bytes = decomposed_safetensors_size(
    reloaded_pruned_model,
    layer_names,
    prefix="_wrapped_model.",
)
disk_size_ratio = pruned_disk_bytes / base_disk_bytes

{
    "save_dir": str(PRUNED_SAVE_DIR),
    "physical_size_ratio": reloaded_pruned_model.physical_size_ratio,
    "decomposed_disk_size_ratio_vs_base": disk_size_ratio,
    "disk_vs_internal_delta": disk_size_ratio - reloaded_pruned_model.physical_size_ratio,
    "deployed_eval_loss": extract_eval_loss(deploy_metrics),
    "reloaded_eval_loss": extract_eval_loss(reloaded_pruned_metrics),
}


/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'save_dir': 'outputs/notebooks/gpt2_flexrank_demo_gar_0.95',
 'physical_size_ratio': 0.9007446928635953,
 'decomposed_disk_size_ratio_vs_base': 0.9007695333402227,
 'disk_vs_internal_delta': 2.4840476627430164e-05,
 'deployed_eval_loss': 3.45381498336792,
 'reloaded_eval_loss': 3.45381498336792}

### Reload verification
Save the deployed model once more, reload it, and assert that evaluation is unchanged.

In [11]:
VERIFY_SAVE_DIR = SAVE_DIR.parent / f"{SAVE_DIR.name}_verify_pruned_{DEPLOY_SIZE_RATIO:g}"

shutil.rmtree(VERIFY_SAVE_DIR, ignore_errors=True)
loaded_model.save_pretrained(VERIFY_SAVE_DIR)

verified_model = FlexRankModel.from_pretrained(VERIFY_SAVE_DIR)
verified_metrics = evaluate_model(verified_model, splits.val, splits.collator, model_data.proc)

verified_loss = extract_eval_loss(verified_metrics)
deployed_loss = extract_eval_loss(deploy_metrics)
verified_impl_counts = odlinear_impl_counts(verified_model)
expected_gar_layers = sum(isinstance(module, ODLinear) for module in verified_model.modules()) if DEPLOY_MODE is DeployMode.GAR else 0
assert verified_impl_counts[ODImpl.GAR.value] == expected_gar_layers
assert abs(verified_loss - deployed_loss) < 1e-6

{
    "save_dir": str(VERIFY_SAVE_DIR),
    "deployed_eval_loss": deployed_loss,
    "reloaded_eval_loss": verified_loss,
    "reloaded_odlinear_impl_counts": verified_impl_counts,
}


/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/rzaccone/envs/flexrank/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'save_dir': 'outputs/notebooks/gpt2_flexrank_demo_verify_pruned_0.95',
 'deployed_eval_loss': 3.45381498336792,
 'reloaded_eval_loss': 3.45381498336792,
 'reloaded_odlinear_impl_counts': {'gar': 48}}